## tl;dr

- The screenshot's 11-fund diagnostics reproduce independently to displayed precision for the 36 complete month-ends from 2023-07-31 through 2026-06-30.
- Source histories contain no duplicate dates, null NAV values, or non-positive NAV values in the downloaded series.
- Lump-sum and DCA accounting match independent share/unit ledgers.
- The audit found and fixed one material Kelly/VA defect: only aggregate risky exposure was rebalanced, allowing the selected fund mix to drift. The regression oracle now requires fund-level target rebalancing with fees.


## Context & Methods

This notebook audits the metrics shown in the supplied screenshot and the backtest accounting. It deliberately does **not** import quant-compass data, diagnostics, frontier, risk, strategy, or allocation helpers for expected-value calculations.

### Key Assumptions

- Requested window: 2023-07-12 through 2026-07-12 (Asia/Shanghai user context).
- Monthly grain: last observed source value per calendar month.
- Only complete month labels within the requested window are retained, producing 36 rows from 2023-07-31 to 2026-06-30.
- Money-fund NAV is independently synthesized from daily per-10,000-unit income.
- Screenshot values are compared at their displayed precision.


## Data

### Load raw-source histories and profile quality

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path("/home/anbinx/project/quant-compass")
sys.path.insert(0, str(ROOT / "analysis"))
import independent_backtest_audit as audit  # noqa: E402

panel, source_quality = audit.build_monthly_panel()
print("Panel:", panel.index.min().date(), "to", panel.index.max().date(), panel.shape)
source_quality

/home/anbinx/project/quant-compass/backend/.venv/lib/python3.12/site-packages/py_mini_racer/py_mini_racer.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Panel: 2023-07-31 to 2026-06-30 (36, 11)


,code,name,type,rows,duplicate_dates,null_values,first_date,last_date,nonpositive_values
0,016149,中银季季享90天滚动持有中短债债券发起A,债券型-中短债,910,0,0,2022-09-07,2026-07-10,0
1,270023,广发全球精选股票(QDII)人民币A,QDII-普通股票,3822,0,0,2010-08-18,2026-07-09,0
2,007339,易方达沪深300ETF联接C,指数型-股票,1751,0,0,2019-04-26,2026-07-10,0
3,002611,博时黄金ETF联接C,指数型-其他,2463,0,0,2016-05-27,2026-07-10,0
4,007280,摩根日本精选股票(QDII)A,QDII-普通股票,1658,0,0,2019-07-31,2026-07-10,0
5,539002,建信新兴市场混合(QDII)A,QDII-混合偏股,3468,0,0,2011-06-21,2026-07-09,0
6,457001,国富亚洲机会股票(QDII)A,QDII-普通股票,3468,0,0,2012-02-22,2026-07-09,0
7,012643,招商中证红利ETF联接A,指数型-股票,1064,0,0,2022-02-23,2026-07-10,0
8,001211,中欧滚钱宝货币A,货币型-普通货币,4017,0,0,2015-06-14,2026-07-12,0
9,006662,易方达安悦超短债A,债券型-中短债,1833,0,0,2018-12-05,2026-07-10,0


## Results

### Independently recompute displayed metrics

In [2]:
metrics = audit.independent_metric_table(panel)
frontier, frontier_summary = audit.independent_frontier(panel)
comparison = audit.compare_to_screenshot(metrics, frontier_summary)
metrics.style.format("{:.4%}")

,cumulative_return,cagr,arithmetic_annual_return,optimizer_return,volatility
016149,11.4570%,3.7901%,3.6280%,7.7037%,1.1790%
270023,162.7545%,39.2798%,34.8397%,27.9913%,21.6137%
007339,29.9097%,9.3891%,10.0249%,11.8617%,16.4072%
002611,81.6955%,22.7281%,21.7230%,19.4655%,18.4247%
007280,43.9481%,13.3073%,13.1827%,13.9143%,13.8907%
539002,263.6015%,55.6937%,48.0152%,36.5554%,30.0945%
457001,142.4963%,35.4995%,34.0126%,27.4537%,30.0811%
012643,-4.0373%,-1.4034%,-0.4557%,5.0493%,13.6693%
001211,4.2150%,1.4260%,1.3771%,6.2406%,0.1166%
006662,0.0197%,0.0068%,0.0102%,5.3521%,0.8471%


### Reconcile against screenshot

In [3]:
delta = pd.DataFrame(index=comparison.index)
for metric in ["cum", "cagr", "optimizer", "vol", "max_weight"]:
    delta[metric + "_delta"] = (
        comparison[metric + "_independent"] - comparison[metric + "_screenshot"]
    )
delta["frontier_usage_delta"] = (
    comparison["used_independent"] - comparison["used_screenshot"]
)
print(
    "Maximum absolute percentage-point delta:",
    delta.drop(columns=["frontier_usage_delta"]).abs().max().max(),
)
print("Maximum frontier usage-count delta:", delta["frontier_usage_delta"].abs().max())
comparison

Maximum absolute percentage-point delta: 4.9925846825824016e-05
Maximum frontier usage-count delta: 0


,cum_independent,cum_screenshot,cagr_independent,cagr_screenshot,optimizer_independent,optimizer_screenshot,vol_independent,vol_screenshot,used_independent,used_screenshot,max_weight_independent,max_weight_screenshot
code,,,,,,,,,,,,
016149,0.114570,0.1146,0.037901,0.0379,0.077037,0.0770,0.011790,0.0118,14,14,0.500000,0.5000
270023,1.627545,1.6275,0.392798,0.3928,0.279913,0.2799,0.216137,0.2161,18,18,0.500000,0.5000
007339,0.299097,0.2991,0.093891,0.0939,0.118617,0.1186,0.164072,0.1641,6,6,0.034488,0.0345
002611,0.816955,0.8170,0.227281,0.2273,0.194655,0.1947,0.184247,0.1842,19,19,0.453840,0.4538
007280,0.439481,0.4395,0.133073,0.1331,0.139143,0.1391,0.138907,0.1389,6,6,0.055888,0.0559
539002,2.636015,2.6360,0.556937,0.5569,0.365554,0.3656,0.300945,0.3009,19,19,0.500000,0.5000
457001,1.424963,1.4250,0.354995,0.3550,0.274537,0.2745,0.300811,0.3008,1,1,0.013211,0.0132
012643,-0.040373,-0.0404,-0.014034,-0.0140,0.050493,0.0505,0.136693,0.1367,1,1,0.012312,0.0123
001211,0.042150,0.0422,0.014260,0.0143,0.062406,0.0624,0.001166,0.0012,7,7,0.246238,0.2462


### Execute independent backtest accounting oracles

In [4]:
import subprocess

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        str(ROOT / "backend" / "tests" / "test_independent_backtest_oracle.py"),
        "-q",
    ],
    cwd=ROOT / "backend",
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

....                                                                     [100%]
4 passed in 0.49s



## Takeaways

1. The screenshot is numerically reproducible from public source histories; the diagnostic calculations and 20-point efficient frontier are correct for the stated window.
2. The underlying source panel is suitable for this audit at monthly grain. The most recent partial July data is intentionally excluded because its month-end label falls after the requested end date.
3. Lump-sum and DCA metrics reconcile with independent cashflow-neutral ledgers.
4. Kelly/VA previously failed the selected-target invariant for multiple funds. A fund could remain substantially overweight whenever aggregate risky exposure was on target. The corrected implementation sells fund-level overweights first and directs available cash to fund-level shortfalls while respecting fees, reserves, and purchase caps.
5. The regression oracle is now the durable acceptance gate for these accounting semantics.
